In [5]:
import importlib
import subprocess
import sys

package = "torch_geometric"

if importlib.util.find_spec(package) is None:
    print(f"{package} not found. Installing...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "torch-geometric"]
    )
    print(f"{package} installed successfully.")
else:
    print(f"{package} is already installed.")

import torch_geometric
print("torch_geometric version:", torch_geometric.__version__)

torch_geometric is already installed.
torch_geometric version: 2.8.0.post1


## GNN for labeling paper citation graph

torch_geometric.datasets is a module in PyTorch Geometric (PyG) that provides ready-to-use graph datasets for node classification, graph classification, link prediction, recommendation systems, molecular property prediction, and more.

In [14]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


In [34]:
# Load Cora dataset
dataset = Planetoid(root="../../Data/Cora", name="Cora")
data = dataset[0].to(device)

print(data)

# Example output:
# Data(x=[2708,1433], edge_index=[2,10556], y=[2708])

## Nodes = scientific papers (~2,708) each described by (1,433 bag of words features) and with a label (out of 7 classes)
## Edges = citation links (~5,400)
## every citation becomes two entries in edge_index.



Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [36]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(
            dataset.num_node_features, 16
        )

        self.conv2 = GCNConv(
            16, dataset.num_classes
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [40]:


model = GCN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

for epoch in range(200):
    model.train()

    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(
            f"Epoch {epoch:3d} "
            f"Loss={loss.item():.4f}"
        )

# Evaluation
model.eval()

pred = model(
    data.x,
    data.edge_index
).argmax(dim=1)

correct = (
    pred[data.test_mask]
    == data.y[data.test_mask]
)

acc = int(correct.sum()) / int(data.test_mask.sum())

print(f"Test Accuracy: {acc:.4f}")

Epoch   0 Loss=1.9421
Epoch  20 Loss=0.2267
Epoch  40 Loss=0.0526
Epoch  60 Loss=0.0527
Epoch  80 Loss=0.0484
Epoch 100 Loss=0.0424
Epoch 120 Loss=0.0429
Epoch 140 Loss=0.0454
Epoch 160 Loss=0.0270
Epoch 180 Loss=0.0327
Test Accuracy: 0.8040


## Money laundering detection example

Elliptic dataset from kaggle

Node represents a single blockchain transaction (amount, fee, inputs..)


An edge 1001 ---> 1002 means that one of the outputs produced by transaction 1001 was later consumed as an input to transaction 1002.





In [43]:
import pandas as pd
import torch

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

import pandas as pd


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [45]:
features = pd.read_csv(
    "../../Data/elliptic_bitcoin_dataset/elliptic_txs_features.csv",
    header=None
)

classes = pd.read_csv(
    "../../Data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv"
)

edges = pd.read_csv(
    "../../Data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"
)

tx_ids = features[0].values

id_map = {
    tx_id: idx
    for idx, tx_id in enumerate(tx_ids)
}

x = torch.tensor(
    features.iloc[:, 1:].values,
    dtype=torch.float
)

# only some transactions are labeled
# 1 = illicit
# 2 = licit
# 3 = ignore


label_map = {}

for _, row in classes.iterrows():

    if row["class"] == "1":
        label_map[row["txId"]] = 1

    elif row["class"] == "2":
        label_map[row["txId"]] = 0

    else:
        label_map[row["txId"]] = -1
        
y = torch.full(
    (len(tx_ids),),
    -1,
    dtype=torch.long
)

for tx_id, idx in id_map.items():
    if tx_id in label_map:
        y[idx] = label_map[tx_id]

edge_list = []

for _, row in edges.iterrows():

    src = id_map[row["txId1"]]
    dst = id_map[row["txId2"]]

    edge_list.append([src, dst])

edge_index = (
    torch.tensor(edge_list)
    .t()
    .contiguous()
)

data = Data(
    x=x,
    edge_index=edge_index,
    y=y
).to(device)

data

In [49]:
labeled_mask = y >= 0
## consider only labeled transactions

indices = labeled_mask.nonzero().view(-1)

n_train = int(0.7 * len(indices))

print("# total transactions=", len(y) , "# labeled transactions=", len(indices))

perm = indices[torch.randperm(len(indices))]

train_idx = perm[:n_train]
test_idx = perm[n_train:]

train_mask = torch.zeros(
    data.num_nodes,
    dtype=torch.bool
)

test_mask = torch.zeros(
    data.num_nodes,
    dtype=torch.bool
)

train_mask[train_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.test_mask = test_mask

# total transactions= 203769 # labeled transactions= 46564


In [51]:
class AML_GCN(torch.nn.Module):

    def __init__(self, in_dim,hidden=128):
        super().__init__()

        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, int(hidden/2))

        self.classifier = torch.nn.Linear(
            int(hidden/2),
            2
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        return self.classifier(x)



In [61]:
model = AML_GCN(
    data.num_features, hidden=24
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs=1000

for epoch in range(num_epochs):

    model.train()

    optimizer.zero_grad()

    out = model(
        data.x,
        data.edge_index
    )

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()
    optimizer.step()

    if epoch%50==0:
        print("Epoch=", epoch,"Loss=",  loss.item()    )

Epoch= 0 Loss= 0.6995095014572144
Epoch= 50 Loss= 0.24078482389450073
Epoch= 100 Loss= 0.20677417516708374
Epoch= 150 Loss= 0.17362475395202637
Epoch= 200 Loss= 0.1516294926404953
Epoch= 250 Loss= 0.13700604438781738
Epoch= 300 Loss= 0.12545330822467804
Epoch= 350 Loss= 0.11594340950250626
Epoch= 400 Loss= 0.10782546550035477
Epoch= 450 Loss= 0.1013742983341217
Epoch= 500 Loss= 0.09611305594444275
Epoch= 550 Loss= 0.09165789932012558
Epoch= 600 Loss= 0.08747009932994843
Epoch= 650 Loss= 0.08364038914442062
Epoch= 700 Loss= 0.08032204210758209
Epoch= 750 Loss= 0.07764007896184921
Epoch= 800 Loss= 0.07518419623374939
Epoch= 850 Loss= 0.07290256023406982
Epoch= 900 Loss= 0.0706491693854332
Epoch= 950 Loss= 0.06860578060150146


In [63]:
model.eval()

with torch.no_grad():

    logits = model(
        data.x,
        data.edge_index
    )

    pred = logits.argmax(dim=1)

    acc = (
        pred[data.test_mask]
        ==
        data.y[data.test_mask]
    ).float().mean()

print("Test accuracy:", acc.item())


Test accuracy: 0.9592698812484741
